# RNA 3D Folding Model - Scientific Validation (Tier 2)

This notebook implements Tier 2 validation for the RNA 3D folding model. The focus is on scientific evaluation of model predictions and performance analysis.

**Characteristics:**
- Medium runtime (15-30 minutes)
- Uses substantial subset of data (10-15 sequences)
- Focuses on scientific metrics, prediction quality, and detailed performance analysis

In [ ]:
import os
import sys
import time
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add project root to path
project_root = Path(os.getcwd()).parent.parent
sys.path.insert(0, str(project_root))

# Import project modules
from src.models.rna_folding_model import RNAFoldingModel
from src.data_loading import RNADataset
from src.utils.structure_metrics import compute_rmsd, compute_tm_score, compute_per_residue_rmsd

# Import tier-specific dataset
sys.path.insert(0, str(Path(os.getcwd()).parent / "tier1_technical"))
from tiered_dataset import TieredRNADataset, verify_data_paths

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Configuration

Define validation parameters and model configuration.

In [ ]:
# Validation configuration
CONFIG = {
    # Data configuration
    "data_dir": os.path.join(project_root, "data", "processed"),
    "validation_tier": "tier2",  # Use tier2 configuration
    
    # Model configuration
    "model_config": {
        "d_model": 256,
        "d_feedforward": 1024,
        "num_layers": 4,
        "num_heads": 8,
        "dropout": 0.1,
        "ipa_dropout": 0.1,
        "use_checkpointing": False
    },
    
    # Paths
    "results_dir": os.path.join(project_root, "validation", "tier2_scientific", "results"),
    "checkpoint_path": None,  # Path to model checkpoint (if available)
    
    # Runtime options
    "random_seed": 42,
    "batch_size": 2,
    "num_workers": 2,
    
    # Additional tier2 parameters
    "extended_metrics": True,  # Enable additional scientific metrics
    "sequence_length_analysis": True,  # Analyze performance by sequence length
    "structure_visualization": True,  # Generate structure visualizations
    "save_predictions": True  # Save model predictions for further analysis
}

# Create results directory if it doesn't exist
os.makedirs(CONFIG["results_dir"], exist_ok=True)

# Verify data paths
print("Verifying data paths:")
verify_data_paths(CONFIG["data_dir"])

## 2. Create Validation Dataset

Load validation data using our tier-specific dataset class.

In [ ]:
# Create validation dataset with tier-specific behavior
validation_dataset = TieredRNADataset(
    data_dir=CONFIG["data_dir"],
    tier=CONFIG["validation_tier"]
)

# Get statistics about the dataset
tier_stats = validation_dataset.get_tier_stats()
print(f"\nTier {tier_stats['tier']} - {tier_stats['description']}:")
print(f"Total IDs: {tier_stats['total_ids']}")
print(f"Required features: {tier_stats['required_features']}")
print(f"Optional features: {tier_stats['optional_features']}")
print(f"Allow mock data: {tier_stats['allow_mock_data']}")

print("\nFeature availability:")
for feature, count in tier_stats['feature_counts'].items():
    percentage = tier_stats['feature_percentages'][feature]
    print(f"  {feature}: {count}/{tier_stats['total_ids']} ({percentage:.1f}%)")
    
print(f"\nIDs with all required features: {tier_stats['ids_with_all_required']}/{tier_stats['total_ids']} "
      f"({tier_stats['ids_with_all_required_percent']:.1f}%)")

# Create validation dataloader
validation_loader = torch.utils.data.DataLoader(
    validation_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    collate_fn=validation_dataset.collate_fn
)

## 3. Initialize Model

Create model instance and load checkpoint if available.

In [ ]:
def initialize_model(config, checkpoint_path=None):
    """Initialize model and load checkpoint if available."""
    # Print full model config for debugging
    print("Model configuration:")
    for key, value in config["model_config"].items():
        print(f"  {key}: {value}")
    
    # Initialize the model with all required parameters
    model = RNAFoldingModel(
        d_model=config["model_config"]["d_model"],
        d_feedforward=config["model_config"]["d_feedforward"],
        num_blocks=config["model_config"]["num_layers"],
        num_attention_heads=config["model_config"]["num_heads"],
        dropout=config["model_config"]["dropout"],
        ipa_dropout=config["model_config"]["ipa_dropout"],
        use_checkpointing=config["model_config"]["use_checkpointing"]
    )
    
    if checkpoint_path and os.path.exists(checkpoint_path):
        print(f"Loading checkpoint from {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        print("Initializing model with random weights")
    
    model = model.to(device)
    return model

# Initialize model
model = initialize_model(CONFIG, CONFIG["checkpoint_path"])

## 4. Model Shape Check

Verify input and output tensor shapes with a sample batch.

In [ ]:
def check_model_shapes(model, dataloader):
    """Check model input and output shapes with a sample batch."""
    batch = next(iter(dataloader))
    
    # Move batch to device
    for key in batch:
        if isinstance(batch[key], torch.Tensor):
            batch[key] = batch[key].to(device)
    
    # Set model to eval mode
    model.eval()
    
    # Forward pass
    with torch.no_grad():
        outputs = model(batch)
    
    # Print input shapes
    print("\nInput shapes:")
    for key, value in batch.items():
        if isinstance(value, torch.Tensor):
            print(f"  {key}: {value.shape}")
    
    # Print output shapes
    print("\nOutput shapes:")
    for key, value in outputs.items():
        if isinstance(value, torch.Tensor):
            print(f"  {key}: {value.shape}")
            
    # Return batch and outputs for further analysis
    return batch, outputs

batch, outputs = check_model_shapes(model, validation_loader)

## 5. Scientific Structure Metrics Evaluation

Evaluate model predictions with expanded scientific metrics.

In [ ]:
def evaluate_extended_metrics(model, dataloader):
    """Evaluate model predictions with expanded scientific metrics."""
    model.eval()
    
    results = {
        "rmsd": [],
        "tm_score": [],
        "per_residue_rmsd": [],
        "ids": [],
        "sequence_lengths": [],
        "predictions": [],  # Store model predictions for further analysis
        "ground_truth": []  # Store ground truth coordinates
    }
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            # Move batch to device
            for key in batch:
                if isinstance(batch[key], torch.Tensor):
                    batch[key] = batch[key].to(device)
            
            # Forward pass
            outputs = model(batch)
            
            # Get predictions and true coordinates - handle different key names
            pred_coords = None
            if "final_atom_positions" in outputs:
                pred_coords = outputs["final_atom_positions"]
            elif "pred_coords" in outputs:
                pred_coords = outputs["pred_coords"]
            else:
                print("Warning: No coordinate predictions found in model outputs")
                continue
                
            true_coords = batch["atom_positions"]
            atom_mask = batch["atom_mask"]
            
            # Store predictions and ground truth for later analysis
            for i in range(len(batch["ids"])):
                mask_i = atom_mask[i] if atom_mask is not None else torch.ones_like(pred_coords[i, :, 0], dtype=torch.bool)
                seq_len = mask_i.sum().item()
                
                results["predictions"].append(pred_coords[i, :seq_len].detach().cpu())
                results["ground_truth"].append(true_coords[i, :seq_len].detach().cpu())
            
            # Calculate RMSD
            rmsd = compute_rmsd(pred_coords, true_coords, atom_mask)
            
            # Calculate TM-score
            tm_score = compute_tm_score(pred_coords, true_coords, atom_mask)
            
            # Calculate per-residue RMSD
            per_res_rmsd = compute_per_residue_rmsd(pred_coords, true_coords, atom_mask)
            
            # Store results
            for i in range(len(batch["ids"])):
                results["ids"].append(batch["ids"][i])
                results["rmsd"].append(rmsd[i].item() if not torch.isnan(rmsd[i]).item() else float('nan'))
                results["tm_score"].append(tm_score[i].item() if not torch.isnan(tm_score[i]).item() else float('nan'))
                results["per_residue_rmsd"].append(per_res_rmsd[i].detach().cpu().numpy())
                if atom_mask is not None:
                    results["sequence_lengths"].append(int(atom_mask[i].sum().item()))
                else:
                    results["sequence_lengths"].append(pred_coords.shape[1])
    
    # Print summary
    print("\nStructure Metrics Summary:")
    rmsd_values = [r for r in results['rmsd'] if not np.isnan(r)]
    tm_values = [r for r in results['tm_score'] if not np.isnan(r)]
    
    if rmsd_values:
        print(f"  Mean RMSD: {np.mean(rmsd_values):.4f} Å")
        print(f"  Median RMSD: {np.median(rmsd_values):.4f} Å")
        print(f"  Min RMSD: {np.min(rmsd_values):.4f} Å")
        print(f"  Max RMSD: {np.max(rmsd_values):.4f} Å")
    else:
        print("  Mean RMSD: N/A (all values are NaN)")
        
    if tm_values:
        print(f"  Mean TM-score: {np.mean(tm_values):.4f}")
        print(f"  Median TM-score: {np.median(tm_values):.4f}")
        print(f"  Min TM-score: {np.min(tm_values):.4f}")
        print(f"  Max TM-score: {np.max(tm_values):.4f}")
    else:
        print("  Mean TM-score: N/A (all values are NaN)")
    
    # Create detailed results table
    print("\nDetailed Results:")
    print(f"{'ID':<15} {'Length':<10} {'RMSD (Å)':<12} {'TM-score':<12}")
    print("-" * 50)
    for i in range(len(results["ids"])):
        rmsd_val = results['rmsd'][i]
        tm_val = results['tm_score'][i]
        rmsd_str = f"{rmsd_val:<12.4f}" if not np.isnan(rmsd_val) else "N/A         "
        tm_str = f"{tm_val:<12.4f}" if not np.isnan(tm_val) else "N/A         "
        print(f"{results['ids'][i]:<15} {results['sequence_lengths'][i]:<10} {rmsd_str} {tm_str}")
    
    # Save results if requested
    if CONFIG["save_predictions"]:
        predictions_dir = os.path.join(CONFIG["results_dir"], "predictions")
        os.makedirs(predictions_dir, exist_ok=True)
        
        for i, target_id in enumerate(results["ids"]):
            pred_file = os.path.join(predictions_dir, f"{target_id}_pred.npy")
            truth_file = os.path.join(predictions_dir, f"{target_id}_true.npy")
            
            np.save(pred_file, results["predictions"][i].numpy())
            np.save(truth_file, results["ground_truth"][i].numpy())
            
        print(f"\nSaved predictions and ground truth to {predictions_dir}")
    
    return results

# Evaluate structure metrics
structure_metrics = evaluate_extended_metrics(model, validation_loader)

## 6. Performance Analysis by Sequence Length

Analyze model performance based on sequence length.

In [ ]:
def analyze_by_sequence_length(results):
    """Analyze model performance based on sequence length."""
    if not CONFIG["sequence_length_analysis"]:
        print("Sequence length analysis disabled in configuration.")
        return
    
    # Create lists for analysis
    lengths = results["sequence_lengths"]
    rmsd_values = results["rmsd"]
    tm_values = results["tm_score"]
    
    # Filter out NaN values
    valid_indices = [i for i, v in enumerate(rmsd_values) if not np.isnan(v)]
    valid_lengths = [lengths[i] for i in valid_indices]
    valid_rmsd = [rmsd_values[i] for i in valid_indices]
    valid_tm = [tm_values[i] for i in valid_indices]
    
    if not valid_lengths:
        print("No valid data for sequence length analysis.")
        return
    
    # Create figure for analysis
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # RMSD vs Sequence Length
    ax1.scatter(valid_lengths, valid_rmsd, alpha=0.7, s=60)
    
    # Add regression line if we have enough points
    if len(valid_lengths) > 2:
        z = np.polyfit(valid_lengths, valid_rmsd, 1)
        p = np.poly1d(z)
        ax1.plot(valid_lengths, p(valid_lengths), "r--", alpha=0.7,
                label=f"Trend: y={z[0]:.4f}x + {z[1]:.4f}")
        
    ax1.set_xlabel("Sequence Length")
    ax1.set_ylabel("RMSD (Å)")
    ax1.set_title("RMSD vs Sequence Length")
    ax1.grid(alpha=0.3)
    if len(valid_lengths) > 2:
        ax1.legend()
    
    # TM-score vs Sequence Length
    ax2.scatter(valid_lengths, valid_tm, alpha=0.7, s=60)
    
    # Add regression line if we have enough points
    if len(valid_lengths) > 2:
        z = np.polyfit(valid_lengths, valid_tm, 1)
        p = np.poly1d(z)
        ax2.plot(valid_lengths, p(valid_lengths), "r--", alpha=0.7,
                label=f"Trend: y={z[0]:.4f}x + {z[1]:.4f}")
        
    ax2.set_xlabel("Sequence Length")
    ax2.set_ylabel("TM-score")
    ax2.set_title("TM-score vs Sequence Length")
    ax2.grid(alpha=0.3)
    if len(valid_lengths) > 2:
        ax2.legend()
    
    plt.tight_layout()
    
    # Save figure
    fig_path = os.path.join(CONFIG["results_dir"], "sequence_length_analysis.png")
    plt.savefig(fig_path)
    print(f"\nSaved sequence length analysis to {fig_path}")
    
    # Additional statistics
    print("\nSequence Length Statistics:")
    print(f"  Minimum length: {min(valid_lengths)}")
    print(f"  Maximum length: {max(valid_lengths)}")
    print(f"  Mean length: {np.mean(valid_lengths):.2f}")
    print(f"  Median length: {np.median(valid_lengths)}")
    
    # Correlation analysis
    rmsd_corr = np.corrcoef(valid_lengths, valid_rmsd)[0, 1]
    tm_corr = np.corrcoef(valid_lengths, valid_tm)[0, 1]
    print("\nCorrelation Analysis:")
    print(f"  Length vs RMSD correlation: {rmsd_corr:.4f}")
    print(f"  Length vs TM-score correlation: {tm_corr:.4f}")
    
    return fig

# Analyze performance by sequence length
sequence_analysis = analyze_by_sequence_length(structure_metrics)

## 7. Visualize Per-Residue RMSD Distribution

Create enhanced visualizations of per-residue RMSD for scientific analysis.

In [ ]:
def visualize_per_residue_rmsd_enhanced(results):
    """Create enhanced visualizations of per-residue RMSD."""
    num_sequences = len(results["ids"])
    
    # Create directory for individual plots
    individual_plots_dir = os.path.join(CONFIG["results_dir"], "per_residue_plots")
    os.makedirs(individual_plots_dir, exist_ok=True)
    
    # Store flattened RMSD values for overall distribution
    all_rmsd_values = []
    
    # Create individual plots
    for i in range(num_sequences):
        per_res_rmsd = results["per_residue_rmsd"][i]
        seq_id = results["ids"][i]
        seq_length = results["sequence_lengths"][i]
        
        # Handle NaN values
        valid_mask = ~np.isnan(per_res_rmsd)
        valid_indices = np.where(valid_mask)[0]
        valid_values = per_res_rmsd[valid_mask]
        
        if len(valid_values) > 0:
            # Store for overall distribution
            all_rmsd_values.extend(valid_values)
            
            # Create individual plot
            fig, ax = plt.subplots(figsize=(10, 5))
            
            # Plot basic line
            ax.plot(valid_indices + 1, valid_values, marker='o', linestyle='-', alpha=0.7)
            
            # Add rolling average for trend
            window_size = min(5, len(valid_values))
            if window_size > 1:
                rolling_avg = np.convolve(valid_values, np.ones(window_size)/window_size, mode='valid')
                ax.plot(valid_indices[window_size-1:] + 1, rolling_avg, 'r-', linewidth=2, 
                      label=f'Rolling average (window={window_size})')
            
            # Show mean line
            mean_rmsd = np.mean(valid_values)
            ax.axhline(y=mean_rmsd, color='g', linestyle='--', 
                      label=f'Mean RMSD: {mean_rmsd:.2f} Å')
            
            # Add quantile lines
            q1, q3 = np.percentile(valid_values, [25, 75])
            ax.axhline(y=q1, color='b', linestyle=':', alpha=0.5,
                      label=f'25th percentile: {q1:.2f} Å')
            ax.axhline(y=q3, color='m', linestyle=':', alpha=0.5,
                      label=f'75th percentile: {q3:.2f} Å')
            
            ax.set_xlabel('Residue Position')
            ax.set_ylabel('RMSD (Å)')
            ax.set_title(f'Per-Residue RMSD for {seq_id} (Length: {seq_length})')
            ax.grid(True, alpha=0.3)
            ax.legend()
            
            # Save individual plot
            fig_path = os.path.join(individual_plots_dir, f"{seq_id}_per_residue_rmsd.png")
            plt.savefig(fig_path)
            plt.close(fig)
    
    # Create overall distribution plot if we have data
    if all_rmsd_values:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Histogram
        sns.histplot(all_rmsd_values, bins=30, kde=True, ax=ax1)
        ax1.set_xlabel('RMSD (Å)')
        ax1.set_ylabel('Frequency')
        ax1.set_title('Distribution of Per-Residue RMSD Values')
        
        # Add statistics
        stats_text = f"Mean: {np.mean(all_rmsd_values):.2f} Å\n"
        stats_text += f"Median: {np.median(all_rmsd_values):.2f} Å\n"
        stats_text += f"Std Dev: {np.std(all_rmsd_values):.2f} Å\n"
        stats_text += f"Min: {np.min(all_rmsd_values):.2f} Å\n"
        stats_text += f"Max: {np.max(all_rmsd_values):.2f} Å\n"
        
        percentiles = [5, 25, 50, 75, 95]
        perc_values = np.percentile(all_rmsd_values, percentiles)
        for p, v in zip(percentiles, perc_values):
            stats_text += f"{p}th percentile: {v:.2f} Å\n"
            
        ax1.text(0.95, 0.95, stats_text, transform=ax1.transAxes, 
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
        
        # Box plot by sequence
        box_data = []
        box_labels = []
        
        for i in range(num_sequences):
            per_res_rmsd = results["per_residue_rmsd"][i]
            seq_id = results["ids"][i]
            
            # Filter out NaN values
            valid_values = per_res_rmsd[~np.isnan(per_res_rmsd)]
            
            if len(valid_values) > 0:
                box_data.append(valid_values)
                box_labels.append(seq_id)
        
        if box_data:
            ax2.boxplot(box_data, labels=box_labels, vert=True)
            ax2.set_xticklabels(box_labels, rotation=45, ha='right')
            ax2.set_ylabel('RMSD (Å)')
            ax2.set_title('Per-Residue RMSD by Sequence')
            ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        # Save overall distribution plot
        dist_path = os.path.join(CONFIG["results_dir"], "rmsd_distribution.png")
        plt.savefig(dist_path)
        print(f"\nSaved RMSD distribution analysis to {dist_path}")
        print(f"Individual per-residue plots saved to {individual_plots_dir}")
        
        return fig
    else:
        print("No valid RMSD values available for visualization.")
        return None

# Visualize per-residue RMSD with enhanced plots
rmsd_visualization = visualize_per_residue_rmsd_enhanced(structure_metrics)

## 8. Memory and Performance Benchmarking

Detailed measurement of memory usage, inference time, and model performance.

In [ ]:
def benchmark_model_detailed(model, dataloader, num_runs=5):
    """Benchmark model memory usage and inference time with detailed metrics."""
    model.eval()
    
    # Get a batch for benchmarking
    batch = next(iter(dataloader))
    for key in batch:
        if isinstance(batch[key], torch.Tensor):
            batch[key] = batch[key].to(device)
    
    # Get batch size and sequence lengths
    batch_size = len(batch["ids"])
    sequence_lengths = []
    if "atom_mask" in batch and batch["atom_mask"] is not None:
        sequence_lengths = [int(batch["atom_mask"][i].sum().item()) for i in range(batch_size)]
    else:
        sequence_lengths = [batch["sequence_int"].shape[1]] * batch_size
    
    # Measure inference time
    times = []
    
    print("\nBenchmarking model inference time...")
    with torch.no_grad():
        # Warmup run
        _ = model(batch)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        
        # Timed runs
        for run in range(num_runs):
            start_time = time.time()
            _ = model(batch)
            torch.cuda.synchronize() if torch.cuda.is_available() else None
            end_time = time.time()
            times.append(end_time - start_time)
    
    avg_time = np.mean(times)
    std_time = np.std(times)
    min_time = np.min(times)
    max_time = np.max(times)
    
    # Calculate throughput
    avg_seq_len = np.mean(sequence_lengths)
    samples_per_second = batch_size / avg_time
    residues_per_second = (batch_size * avg_seq_len) / avg_time
    
    # Measure memory usage
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.empty_cache()
        
        # Forward pass to measure memory
        with torch.no_grad():
            _ = model(batch)
        
        # Get memory stats
        peak_memory = torch.cuda.max_memory_allocated(device) / (1024 ** 2)  # MB
        memory_str = f"{peak_memory:.2f} MB"
        
        # Calculate memory per sample
        memory_per_sample = peak_memory / batch_size
        memory_per_residue = peak_memory / (batch_size * avg_seq_len)
    else:
        memory_str = "N/A (CPU only)"
        memory_per_sample = None
        memory_per_residue = None
    
    print(f"\nDetailed Performance Metrics:")
    print(f"  Batch size: {batch_size}")
    print(f"  Average sequence length: {avg_seq_len:.1f} residues")
    print(f"\nInference Time:")
    print(f"  Average: {avg_time:.4f} ± {std_time:.4f} seconds")
    print(f"  Range: {min_time:.4f} - {max_time:.4f} seconds")
    print(f"  Samples per second: {samples_per_second:.2f}")
    print(f"  Residues per second: {residues_per_second:.2f}")
    
    print(f"\nMemory Usage:")
    print(f"  Peak memory: {memory_str}")
    if memory_per_sample is not None:
        print(f"  Memory per sample: {memory_per_sample:.2f} MB")
        print(f"  Memory per residue: {memory_per_residue:.2f} MB")
    
    # Save results
    benchmark_results = {
        "inference_time": {
            "mean": avg_time,
            "std": std_time,
            "min": min_time,
            "max": max_time,
            "unit": "seconds"
        },
        "throughput": {
            "samples_per_second": samples_per_second,
            "residues_per_second": residues_per_second
        },
        "batch_size": batch_size,
        "average_sequence_length": float(avg_seq_len),
        "peak_memory": memory_str,
        "memory_per_sample": memory_per_sample,
        "memory_per_residue": memory_per_residue
    }
    
    return benchmark_results

# Benchmark model
benchmark_results = benchmark_model_detailed(model, validation_loader)

## 9. Save Validation Results

Save comprehensive validation results to a JSON file for further analysis.

In [ ]:
def save_validation_results(structure_metrics, benchmark_results):
    """Save validation results to a JSON file."""
    # Prepare results for saving
    
    # Filter out NaN values for statistics
    rmsd_values = [r for r in structure_metrics["rmsd"] if not np.isnan(r)]
    tm_values = [r for r in structure_metrics["tm_score"] if not np.isnan(r)]
    
    results = {
        "validation_time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "device": str(device),
        "config": CONFIG,
        "structure_metrics": {
            "rmsd": {
                "mean": float(np.mean(rmsd_values)) if rmsd_values else None,
                "median": float(np.median(rmsd_values)) if rmsd_values else None,
                "std": float(np.std(rmsd_values)) if rmsd_values else None,
                "min": float(np.min(rmsd_values)) if rmsd_values else None,
                "max": float(np.max(rmsd_values)) if rmsd_values else None,
                "per_sequence": [
                    {
                        "id": id, 
                        "length": length, 
                        "rmsd": float(rmsd) if not np.isnan(rmsd) else None
                    } 
                    for id, length, rmsd in zip(
                        structure_metrics["ids"], 
                        structure_metrics["sequence_lengths"], 
                        structure_metrics["rmsd"]
                    )
                ]
            },
            "tm_score": {
                "mean": float(np.mean(tm_values)) if tm_values else None,
                "median": float(np.median(tm_values)) if tm_values else None,
                "std": float(np.std(tm_values)) if tm_values else None,
                "min": float(np.min(tm_values)) if tm_values else None,
                "max": float(np.max(tm_values)) if tm_values else None,
                "per_sequence": [
                    {
                        "id": id, 
                        "length": length, 
                        "tm_score": float(tm_score) if not np.isnan(tm_score) else None
                    } 
                    for id, length, tm_score in zip(
                        structure_metrics["ids"], 
                        structure_metrics["sequence_lengths"], 
                        structure_metrics["tm_score"]
                    )
                ]
            },
            # Per-residue RMSD summary (only means to save space)
            "per_residue_rmsd": {
                "per_sequence": [
                    {
                        "id": id, 
                        "length": length, 
                        "mean_rmsd": float(np.nanmean(per_res_rmsd)) if not np.all(np.isnan(per_res_rmsd)) else None,
                        "median_rmsd": float(np.nanmedian(per_res_rmsd)) if not np.all(np.isnan(per_res_rmsd)) else None,
                        "min_rmsd": float(np.nanmin(per_res_rmsd)) if not np.all(np.isnan(per_res_rmsd)) else None,
                        "max_rmsd": float(np.nanmax(per_res_rmsd)) if not np.all(np.isnan(per_res_rmsd)) else None
                    } 
                    for id, length, per_res_rmsd in zip(
                        structure_metrics["ids"], 
                        structure_metrics["sequence_lengths"], 
                        structure_metrics["per_residue_rmsd"]
                    )
                ]
            }
        },
        "benchmark_results": benchmark_results
    }
    
    # Add sequence length analysis if enabled
    if CONFIG["sequence_length_analysis"] and rmsd_values and len(structure_metrics["sequence_lengths"]) > 1:
        # Filter out NaN values for correlation
        valid_indices = [i for i, v in enumerate(structure_metrics["rmsd"]) if not np.isnan(v)]
        valid_lengths = [structure_metrics["sequence_lengths"][i] for i in valid_indices]
        valid_rmsd = [structure_metrics["rmsd"][i] for i in valid_indices]
        valid_tm = [structure_metrics["tm_score"][i] for i in valid_indices]
        
        if valid_lengths and len(valid_lengths) > 1:
            rmsd_corr = float(np.corrcoef(valid_lengths, valid_rmsd)[0, 1])
            tm_corr = float(np.corrcoef(valid_lengths, valid_tm)[0, 1])
            
            results["sequence_length_analysis"] = {
                "length_stats": {
                    "min": int(min(valid_lengths)),
                    "max": int(max(valid_lengths)),
                    "mean": float(np.mean(valid_lengths)),
                    "median": float(np.median(valid_lengths))
                },
                "correlations": {
                    "length_vs_rmsd": rmsd_corr,
                    "length_vs_tm_score": tm_corr
                }
            }
    
    # Custom JSON encoder to handle NaN values
    class NpEncoder(json.JSONEncoder):
        def default(self, obj):
            if isinstance(obj, np.integer):
                return int(obj)
            if isinstance(obj, np.floating):
                return float(obj)
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            if np.isnan(obj):
                return None
            return super(NpEncoder, self).default(obj)
    
    # Save to file
    results_file = os.path.join(CONFIG["results_dir"], "validation_results.json")
    with open(results_file, 'w') as f:
        json.dump(results, f, indent=2, cls=NpEncoder)
    
    print(f"\nValidation results saved to {results_file}")
    return results

# Save validation results
validation_results = save_validation_results(structure_metrics, benchmark_results)

## 10. Validation Summary

Display a detailed summary of validation results.

In [ ]:
def display_summary(validation_results):
    """Display a detailed summary of validation results."""
    print("\n" + "=" * 60)
    print("RNA 3D Folding Model - Scientific Validation Summary (Tier 2)")
    print("=" * 60)
    print(f"Validation performed on {validation_results['validation_time']}")
    print(f"Device: {validation_results['device']}")
    print(f"Number of sequences evaluated: {len(validation_results['structure_metrics']['rmsd']['per_sequence'])}")
    
    # Structure metrics
    print("\nStructure Metrics:")
    
    # Handle potentially missing metric values
    rmsd_mean = validation_results['structure_metrics']['rmsd']['mean']
    rmsd_median = validation_results['structure_metrics']['rmsd']['median']
    rmsd_min = validation_results['structure_metrics']['rmsd']['min']
    rmsd_max = validation_results['structure_metrics']['rmsd']['max']
    
    tm_mean = validation_results['structure_metrics']['tm_score']['mean']
    tm_median = validation_results['structure_metrics']['tm_score']['median']
    tm_min = validation_results['structure_metrics']['tm_score']['min']
    tm_max = validation_results['structure_metrics']['tm_score']['max']
    
    if rmsd_mean is not None:
        print(f"  RMSD:")
        print(f"    Mean: {rmsd_mean:.4f} Å")
        print(f"    Median: {rmsd_median:.4f} Å")
        print(f"    Range: {rmsd_min:.4f} - {rmsd_max:.4f} Å")
    else:
        print("  RMSD: N/A (no valid values)")
        
    if tm_mean is not None:
        print(f"  TM-score:")
        print(f"    Mean: {tm_mean:.4f}")
        print(f"    Median: {tm_median:.4f}")
        print(f"    Range: {tm_min:.4f} - {tm_max:.4f}")
    else:
        print("  TM-score: N/A (no valid values)")
    
    # Performance metrics
    print("\nPerformance Metrics:")
    print(f"  Inference time: {validation_results['benchmark_results']['inference_time']['mean']:.4f} seconds")
    print(f"  Samples per second: {validation_results['benchmark_results']['throughput']['samples_per_second']:.2f}")
    print(f"  Residues per second: {validation_results['benchmark_results']['throughput']['residues_per_second']:.2f}")
    print(f"  Peak memory usage: {validation_results['benchmark_results']['peak_memory']}")
    
    # Sequence length analysis if available
    if 'sequence_length_analysis' in validation_results:
        print("\nSequence Length Analysis:")
        print(f"  Length range: {validation_results['sequence_length_analysis']['length_stats']['min']} - "
              f"{validation_results['sequence_length_analysis']['length_stats']['max']} residues")
        print(f"  Length vs RMSD correlation: {validation_results['sequence_length_analysis']['correlations']['length_vs_rmsd']:.4f}")
        print(f"  Length vs TM-score correlation: {validation_results['sequence_length_analysis']['correlations']['length_vs_tm_score']:.4f}")
    
    # Overall assessment
    print("\nScientific Assessment:")
    if tm_mean is not None:
        if tm_mean > 0.8:
            assessment = "Excellent - Near native-like predictions"
        elif tm_mean > 0.6:
            assessment = "Good - Correct overall fold"
        elif tm_mean > 0.4:
            assessment = "Fair - Partially correct topology"
        else:
            assessment = "Poor - Significant structural errors"
        print(f"  Model quality: {assessment} (based on mean TM-score of {tm_mean:.4f})")
    else:
        print("  Model quality: Cannot be determined (no valid TM-scores)")
    print("=" * 60)

# Display summary
display_summary(validation_results)

## 11. Structure Visualization (Optional)

Visualize predicted structures if visualization modules are available.

In [ ]:
def visualize_structures():
    """Visualize predicted structures if visualization is enabled."""
    if not CONFIG["structure_visualization"]:
        print("Structure visualization is disabled in configuration.")
        return
    
    try:
        import py3Dmol
        from IPython.display import display
        print("Structure visualization is available but requires predicted PDB files.")
        print("To visualize structures, save predictions as PDB files and use py3Dmol.")
        
        # Example visualization code (requires PDB files)
        print("\nExample py3Dmol visualization code:")
        print("""
        # Create viewer
        view = py3Dmol.view(width=800, height=600)
        
        # Load predicted structure (red)
        view.addModel(open('predicted.pdb').read(), 'pdb')
        view.setStyle({'model': 0}, {'cartoon': {'color': 'red'}})
        
        # Load ground truth structure (blue)
        view.addModel(open('ground_truth.pdb').read(), 'pdb')
        view.setStyle({'model': 1}, {'cartoon': {'color': 'blue'}})
        
        # Set view options
        view.zoomTo()
        view.spin(True)
        
        # Display viewer
        view.show()
        """)
    except ImportError:
        print("py3Dmol is not available for structure visualization.")
        print("To enable visualization, install py3Dmol: pip install py3dmol")

# Visualize structures if enabled
visualize_structures()